# Rule: **build_industry_sector_ratios_intermediate**


**Description**

This rule uses the current industrial energy demand by country (in TWh/a), divides it by physical industrial production (tMaterial/a), and reconstructs present‑day energy intensity ratios (MWh/tMaterial). Then it interpolates these values with the future sectoral ratios from the best case scenario (built with `build_industry_sector_ratios.py`) using the parameter `sector_ratios_fraction_future.horizon year` from the configuration file. This process yields intermediate energy intensity ratios (MWh / tMaterial) by country and sector for the horizon year.  

The configuration parameter associated with this rule is defined under the **industry** section of the config file. It represents how close to the ideal scenario the industry is projected to be in the horizon year.

- industry.sector_ratios_fraction_future

**Inputs**

- resources/{prefix}/{name}/`industry_sector_ratios.csv`
- resources/{prefix}/{name}/`industrial_production_per_country.csv`
- resources/{prefix}/{name}/`industrial_energy_demand_per_country_today.csv`

**Outputs**

- resources/{prefix}/{name}/`industry_sector_ratios_{horizon}.csv`

In [ ]:
######################################## Parameters

### Run
prefix = ''
name = ''

### Network
horizon = ''

In [ ]:
##### Imports
import pandas as pd
import os 
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')

##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

##### Set options
pd.set_option("display.max_columns", None)

## `industry_sector_ratios_{horizon}.csv`  
Load the file and preview its content.

In [ ]:
file = f"industry_sector_ratios_{horizon}.csv"

sector_ratios_intermediate = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
    header=1,
    index_col=0,
)
sector_ratios_intermediate = sector_ratios_intermediate.iloc[1:]
sector_ratios_intermediate.index.name = "MWh/tMaterial"
sector_ratios_intermediate

Parameters and layout colors for the graphs

In [ ]:
#################### Parameters

### Industrial sectors to plot. Choose any subset, in desired order.
sectors_to_plot = [
    "Electric arc",
    "DRI + Electric arc",
    "Integrated steelworks",
    "HVC",
    "HVC (mechanical recycling)",
    "HVC (chemical recycling)",
    "Ammonia",
    "Chlorine",
    "Methanol",
    "Other chemicals",
    "Pharmaceutical products etc.",
    "Cement",
    "Ceramics & other NMM",
    "Glass production",
    "Pulp production",
    "Paper production",
    "Printing and media reproduction",
    "Food, beverages and tobacco",
    "Alumina production",
    "Aluminium - primary production",
    "Aluminium - secondary production",
    "Other non-ferrous metals",
    "Transport equipment",
    "Machinery equipment",
    "Textiles and leather",
    "Wood and wood products",
    "Other industrial sectors",
]

energy_carriers = [
    "elec",
    "coal",
    "coke",
    "biomass",
    "methane",
    "hydrogen",
    "heat",
    "naphtha",
    "ammonia",
    "methanol"
]

############ Energy carrier colors
colors = {
    "elec": "#1f77b4",
    "coal": "#444444",
    "coke": "#8c564b",
    "biomass": "#2ca02c",
    "methane": "#ff7f0e",
    "hydrogen": "#17becf",
    "heat": "#d62728",
    "naphtha": "#9467bd",
    "ammonia": "#bcbd22",
    "methanol": "#e377c2"
}

# exclude emissions
exclude_rows = [
    "process emission",
    "process emission from feedstock",
]

How do energy intensities vary across different industrial sectors?

In [ ]:
#################### Prepare data

sector_ratios_energy = sector_ratios_intermediate.drop(
    index=[r for r in exclude_rows if r in sector_ratios_intermediate.index]
)

data = []

for sector in sectors_to_plot:
    for carrier in energy_carriers:
        value = sector_ratios_energy.loc[carrier, sector]
        data.append({
            "sector": sector,
            "carrier": carrier,
            "value": value
        })

df_long = pd.DataFrame(data)
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce").fillna(0)

#################### Reorder data for plotting

df_plot = df_long.pivot(index="sector", columns="carrier", values="value").fillna(0)
df_plot = df_plot.reindex(sectors_to_plot)

carriers_in_plot = [carrier for carrier in energy_carriers if carrier in df_plot.columns]
df_plot = df_plot[carriers_in_plot]

#################### Plot

fig, ax = plt.subplots(figsize=(11, 6))

y = np.arange(len(df_plot.index))
pos_cum = np.zeros(len(df_plot.index))
neg_cum = np.zeros(len(df_plot.index))

for carrier in carriers_in_plot:
    values = df_plot[carrier].values.astype(float)

    left = np.where(values >= 0, pos_cum, neg_cum)

    ax.barh(
        y,
        values,
        left=left,
        color=colors.get(carrier, "#cccccc"),
        edgecolor="none",
        label=carrier
    )

    pos_cum += np.where(values > 0, values, 0)
    neg_cum += np.where(values < 0, values, 0)

#################### Layout

ax.axvline(0, color="black", linewidth=0.8)

ax.set_title(f"Sector ratios in {horizon}")
ax.set_xlabel("Energy ratios (MWh/tMaterial)")
ax.set_ylabel("Industrial sector")

ax.set_yticks(y)
ax.set_yticklabels(df_plot.index)
ax.invert_yaxis()

xmin = neg_cum.min()
xmax = pos_cum.max()
xrange = xmax - xmin

if np.isclose(xrange, 0):
    xrange = 1

pad = 0.05 * xrange
ax.set_xlim(xmin - pad, xmax + pad)

ax.legend(
    title="Energy carrier",
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    frameon=False
)

ax.set_facecolor("white")
fig.patch.set_facecolor("white")

plt.tight_layout()
plt.show()


What is the difference between the estimated energy intensity ratios for the horizon year and for the best case scenario expected in the future?

In [ ]:
##### Load the ideal scenario sector ratios for comparison
file = f"industry_sector_ratios.csv"

sector_ratios = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
    header=0,
    index_col=0,
)

#################### Build data

maps = []

for sector in sectors_to_plot:

    future = sector_ratios[sector].astype(float)
    intermediate = sector_ratios_intermediate[sector].astype(float)

    comparison = pd.concat(
        [intermediate, future],
        axis=1
    )

    comparison.columns = [horizon, "Ideal scenario"]

    comparison = comparison.reindex(energy_carriers).fillna(0.0)

    maps.append((sector, comparison))

#################### Layout

n_cols = 4
n_vars = len(maps)
n_rows = int(np.ceil(n_vars / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4.2 * n_cols, 3.4 * n_rows),
    squeeze=False
)

#################### Plot

for i, (sector, df) in enumerate(maps):

    row = i // n_cols
    col = i % n_cols
    ax = axes[row, col]

    scenarios = [horizon, "Ideal scenario"]
    y = np.arange(len(scenarios))

    pos_cum = np.zeros(len(scenarios))
    neg_cum = np.zeros(len(scenarios))

    for carrier in energy_carriers:
        values = np.array([
            df.loc[carrier, horizon],
            df.loc[carrier, "Ideal scenario"]
        ], dtype=float)

        left = np.where(values >= 0, pos_cum, neg_cum)

        ax.barh(
            y,
            values,
            left=left,
            color=colors.get(carrier, "#cccccc"),
            edgecolor="none",
            height=0.6
        )

        pos_cum += np.where(values > 0, values, 0)
        neg_cum += np.where(values < 0, values, 0)

    #################### Individual x-limits for each subplot

    local_pos_max = pos_cum.max()
    local_neg_min = neg_cum.min()
    local_range = local_pos_max - local_neg_min

    if np.isclose(local_range, 0):
        local_range = 1.0

    local_pad = 0.05 * local_range
    ax.set_xlim(local_neg_min - local_pad, local_pos_max + local_pad)

    #################### Axis styling

    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(sector)
    ax.set_yticks(y)
    ax.set_yticklabels(scenarios)
    ax.invert_yaxis()
    ax.set_facecolor("white")

    if row < n_rows - 1:
        ax.tick_params(axis="x", labelbottom=False)
    else:
        ax.set_xlabel("Energy ratios (MWh/tMaterial)")

# apagar ejes vacíos
for j in range(n_vars, n_rows * n_cols):
    row = j // n_cols
    col = j % n_cols
    axes[row, col].axis("off")

#################### Global title and legend

fig.suptitle(f"Sector ratios {horizon} vs ideal scenario ", fontsize=16, y=0.98)

legend_handles = [
    Patch(facecolor=colors.get(carrier, "#cccccc"), edgecolor="none", label=carrier)
    for carrier in energy_carriers
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=min(5, len(energy_carriers)),
    frameon=False,
    title="Carrier",
    bbox_to_anchor=(0.5, 0.955),
    fontsize=12,
    title_fontsize=13
)

plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.show()